# SMS Spam Detection using Naive Bayes and TF-IDF

This notebook trains an SMS spam classifier using a **public dataset URL**, **TF-IDF**, and **Multinomial Naive Bayes**.

### Workflow
1. Load the dataset directly from a public URL
2. Clean and preprocess SMS text
3. Convert text to TF-IDF features
4. Split the data
5. Train Multinomial Naive Bayes
6. Evaluate the model
7. Test custom SMS messages
8. Save the trained model and vectorizer for the Streamlit app


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 1. Load the public SMS Spam dataset

The dataset is loaded directly from a public URL. No dataset file needs to be uploaded to Colab.

The URL below points to the commonly used **SMS Spam Collection** dataset.


In [ ]:
# Public dataset URL
DATASET_URL = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"

# Load the dataset directly from the public URL
df = pd.read_csv(
    DATASET_URL,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8"
)

print("Dataset shape:", df.shape)
df.head()


In [ ]:
# Basic dataset information
print("Class distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())


## 2. Clean and preprocess the SMS text

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df.dropna(subset=["label", "message"]).copy()
df["clean_message"] = df["message"].apply(clean_text)

df[["label", "message", "clean_message"]].head()


## 3. Convert labels to numeric values

In [ ]:
# ham = 0, spam = 1
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

X = df["clean_message"]
y = df["label_num"]

print(df[["label", "label_num"]].drop_duplicates())


## 4. Split the dataset into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 5. TF-IDF Feature Extraction

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)


## 6. Train the Multinomial Naive Bayes model

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

print("Model training completed successfully.")


## 7. Evaluate the model

In [ ]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"]
))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Ham", "Spam"]
)

disp.plot()
plt.title("SMS Spam Detection - Confusion Matrix")
plt.show()


## 8. Test custom SMS messages

In [ ]:
def predict_sms(message):
    cleaned = clean_text(message)
    features = tfidf.transform([cleaned])

    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    confidence = probabilities[prediction]

    label = "SPAM" if prediction == 1 else "HAM"

    return label, confidence

test_messages = [
    "Congratulations! You have won a free prize. Call now!",
    "Hey, are we meeting for college tomorrow?",
    "URGENT! You have been selected for a cash reward. Click the link now.",
    "Can you send me the assignment notes?"
]

for message in test_messages:
    label, confidence = predict_sms(message)

    print(f"Message: {message}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2%}")
    print("-" * 70)


## 9. Interactive SMS prediction

In [ ]:
user_message = input("Enter an SMS message: ")

label, confidence = predict_sms(user_message)

print(f"\nPrediction: {label}")
print(f"Confidence: {confidence:.2%}")


## 10. Save the trained model and TF-IDF vectorizer

These files can be used by the Streamlit `app.py` so the application does not need to retrain the model every time.


In [ ]:
joblib.dump(model, "spam_classifier.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model saved as: spam_classifier.pkl")
print("Vectorizer saved as: tfidf_vectorizer.pkl")


## Conclusion

The SMS Spam Collection dataset was loaded directly from a public URL. The messages were cleaned, transformed into TF-IDF features, and classified using a Multinomial Naive Bayes model.

The trained model and vectorizer were saved for use in the Streamlit application.
